Install the recorded analysis environment before running this notebook; `scanpy[leiden]` is required.


In [2]:
import os
from pathlib import Path
import scanpy as sc
import numpy as np
from scipy import sparse

# ============================================================
# Load data
# ============================================================
data_root = Path(os.environ.get("DTMP_FIG4J_DATA_DIR", "data")).resolve()
output_root = Path(os.environ.get("DTMP_FIG4J_OUTPUT_DIR", "results")).resolve()
input_path = data_root / "10X_Lung_Tumour_Annotated_v2.h5ad"
if not input_path.is_file():
    raise FileNotFoundError(f"Figure 4J AnnData input missing: {input_path}")
adata = sc.read_h5ad(input_path)

# === Define malignant epithelial types ===
MALIGNANT_EPITHELIAL = [
    "AT2 cells",
    "Cycling AT2 cells",
    "Atypical epithelial cells"
]

# === Filtering: tumour environment + malignant epithelial ===
mask_tumor = adata.obs["environment"] == "Tumour"
mask_epithelial = adata.obs["Cell types v25"].isin(MALIGNANT_EPITHELIAL)
mask_malignant = mask_tumor & mask_epithelial
print(f"Reannotating {mask_malignant.sum()} malignant tumour epithelial cells.")

# ============================================================
# Extract expression values
# ============================================================
X = adata.X
var_names = adata.var_names

# Indices (TYMS/TK1 only)
i_tyms = np.where(var_names == "TYMS")[0][0]
i_tk1  = np.where(var_names == "TK1")[0][0]

# Dense extraction
if sparse.issparse(X):
    tyms = X[:, i_tyms].toarray().ravel()
    tk1  = X[:, i_tk1].toarray().ravel()
else:
    tyms = X[:, i_tyms].ravel()
    tk1  = X[:, i_tk1].ravel()

# Flags
adata.obs['_TYMS_pos'] = tyms > 0
adata.obs['_TK1_pos']  = tk1 > 0

# ============================================================
# TYMS/TK1 state definition (4 combinations)
# ============================================================
choices = [
    'TYMS- TK1-',  # 0
    'TYMS- TK1+',  # 1
    'TYMS+ TK1-',  # 2
    'TYMS+ TK1+'   # 3
]

# Code (binary): TYMS * 2 + TK1 * 1
codes = (
    adata.obs['_TYMS_pos'].astype(int) * 2 +
    adata.obs['_TK1_pos'].astype(int)  * 1
)

# ============================================================
# Assign state to malignant epithelial cells
# ============================================================
adata.obs['Tumour_TYMS_TK1_state'] = "Not tumour epithelial"
adata.obs.loc[mask_malignant, 'Tumour_TYMS_TK1_state'] = codes.map(dict(enumerate(choices)))

# ============================================================
# Update Cell types v25 (same as original logic)
# ============================================================
col = 'Cell types v25'

if str(adata.obs[col].dtype).startswith('category'):
    new_cats = list(adata.obs[col].cat.categories) + choices
    adata.obs[col] = adata.obs[col].cat.set_categories(new_cats)

# Overwrite only malignant epithelial subset
adata.obs.loc[mask_malignant, col] = adata.obs.loc[mask_malignant, 'Tumour_TYMS_TK1_state']


Reannotating 30322 malignant tumour epithelial cells.


In [6]:
import pandas as pd

# --- Basic counts per cell type (all cells) ---
ct_counts_all = adata.obs['Cell types v25'].value_counts()
print("\n=== Cell counts per 'Cell types v25' (ALL cells) ===")
print(ct_counts_all)

# --- Tumour-only counts per cell type ---
mask_tumour = adata.obs["environment"] == "Tumour"
ct_counts_tumour = adata.obs.loc[mask_tumour, 'Cell types v25'].value_counts()
print("\n=== Cell counts per 'Cell types v25' (Tumour only) ===")
print(ct_counts_tumour)

# --- Malignant epithelial definition ---
MALIGNANT_EPITHELIAL = [
    "AT2 cells",
    "Cycling AT2 cells",
    "Atypical epithelial cells"
]
mask_epithelial = adata.obs["Cell types v25"].isin(MALIGNANT_EPITHELIAL)
mask_malignant = mask_tumour & mask_epithelial

print(f"\nTotal malignant tumour epithelial cells: {mask_malignant.sum()}")

# Detailed breakdown of malignant epithelial cell types in tumour
ct_counts_malignant = (
    adata.obs.loc[mask_malignant, 'Cell types v25']
    .value_counts()
    .reindex(MALIGNANT_EPITHELIAL, fill_value=0)
)
print("\n=== Malignant epithelial counts in Tumour ===")
print(ct_counts_malignant)

# OPTIONAL: save a summary table to CSV for record
summary_df = pd.DataFrame({
    "All_cells": ct_counts_all,
    "Tumour_only": ct_counts_tumour
}).fillna(0).astype(int)

output_root.mkdir(parents=True, exist_ok=True)
summary_path = output_root / "Cell_types_v25__classification.csv"
summary_df.to_csv(summary_path)
print(f"\nSaved cell-type count summary to: {summary_path}")



=== Cell counts per 'Cell types v25' (ALL cells) ===
Cell types v25
mo-DC2                                    98738
B cells                                   47023
Anti-inflammatory macrophages             44399
Cytotoxic T cells                         42904
TYMS- TK1-                                24094
Naive T cells                             21447
NK cells (higher cytotoxicity)            18994
Exhausted T cells                         17288
NK cells (lower cytotoxicity)             17052
Downregulated T cells                     16448
Downregulated B cells                     16046
cDC2                                      15117
Pro-inflammatory macrophages              14140
STAB1+ anti-inflammatory macrophages      14059
Tregs                                     12083
Exhausted cytotoxic T cells                9132
Cycling anti-inflammatory macrophages      8519
Plasma B cells                             8234
pDCs                                       7327
Anti-inflammatory a

In [ ]:
from scipy import sparse
import numpy as np

def size_of_matrix(mat):
    if mat is None:
        return 0
    if sparse.issparse(mat):
        return (mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes)
    elif hasattr(mat, "nbytes"):
        return mat.nbytes
    else:
        return 0

print("=== Main matrices ===")
print("X:      ~%.1f MB" % (size_of_matrix(adata.X) / 1024**2))

if adata.raw is not None:
    print("raw.X:  ~%.1f MB" % (size_of_matrix(adata.raw.X) / 1024**2))

print("\n=== Layers ===")
for name, layer in adata.layers.items():
    print(f"layer[{name!r}]: ~{size_of_matrix(layer)/1024**2:.1f} MB")

print("\n=== obsm ===")
for k, v in adata.obsm.items():
    print(f"obsm[{k!r}]: shape={getattr(v, 'shape', None)} ~{size_of_matrix(v)/1024**2:.1f} MB")

print("\n=== obsp (graphs) ===")
for k, v in adata.obsp.items():
    print(f"obsp[{k!r}]: shape={getattr(v, 'shape', None)} ~{size_of_matrix(v)/1024**2:.1f} MB")


In [ ]:
print("Has raw before:", adata.raw is not None)

# Drop raw (this removes raw.X and all raw.var, etc.)
adata.raw = None

print("Has raw after:", adata.raw is not None)

# Save a new version WITHOUT raw
output_root.mkdir(parents=True, exist_ok=True)
out_path = output_root / "10X_Lung_Tumour_Annotated_v2_TYMSTK1_classification.h5ad"
adata.write(out_path, compression="gzip")

print(f"Saved AnnData without raw to:\n  {out_path}")